In [1]:
# Import required libraries
from tuecycle.data.loader import DataManager, get_station
from tuecycle.utils.weather import (
    WeatherCompositeIndex,
    add_weather_index,
    add_weather_quartile,
    estimate_city_elasticity,
)
from tuecycle.plots.functions import (
    plot_weather_index_scatter,
    plot_weather_quartile_boxplot,
    plot_weather_index_hourly,
    plot_city_elasticity_comparison,
    plot_city_weather_sensitivity_heatmap,
    plot_city_resilience_ranking,
)
import pandas as pd
import numpy as np

In [2]:
USE_CITIES = True
WINSORIZE_PERCENTILES = (0.01, 0.99) # or None

START_DATE = (2022, 12, 1)
END_DATE = (2025, 11, 30)
BASE_PATH = ".."  # Since notebooks are in exp/ directory

## 1. Load Data from Multiple Cities

In [3]:
# Initialize DataManager and load data for multiple cities
dm = DataManager(base_path=BASE_PATH, start_date=START_DATE, end_date=END_DATE)

# Select cities from different regions
if USE_CITIES:
    cities = [
        "stuttgart",
        "heidelberg",
        "mannheim",
        "tuebingen",
        "karlsruhe",
        "ludwigsburg",
        "freiburg",
        "ulm",
        "ravensburg",
        "loerrach",
        "konstanz",
        "heilbronn",
        "kirchheim"
    ]

    # Load cities with z-score normalized aggregation
    data_dict = {}
    for city in cities:
        try:
            df = dm.get_city(city, winsorize_percentiles=WINSORIZE_PERCENTILES, force_reload=False)
            data_dict[city] = df
            print(f"✓ Loaded {city}: {len(df):,} hours, {df['bike'].notna().sum():,} with bike data")
        except Exception as e:
            print(f"✗ {city}: {e}")

    print(f"\nTotal: {len(data_dict)} cities loaded")
else:
    stations = [
        "stuttgart_koenig_karls",      # Stuttgart
        "heidelberg_mannheimer", # Heidelberg 
        "mannheim_kurpfalz",     # Mannheim
        "tuebingen_tunnel",      # Tübingen - university town
        "karlsruhe_erbprinzen",  # Karlsruhe
        "ludwigsburg_alleen",    # Ludwigsburg
    ]

    # Load data into dictionary
    data_dict = {}
    for station in stations:
        try:
            df = dm.get(station, force_reload=False)
            data_dict[station] = df
            print(f"✓ Loaded {station}: {len(df):,} hours, {df['bike'].notna().sum():,} with bike data")
        except Exception as e:
            print(f"✗ {station}: {e}")

    print(f"\nTotal: {len(data_dict)} stations loaded")

✓ Loaded stuttgart: 26,304 hours, 26,266 with bike data
✓ Loaded heidelberg: 26,304 hours, 26,247 with bike data
✓ Loaded mannheim: 26,304 hours, 26,266 with bike data
✓ Loaded tuebingen: 26,304 hours, 26,228 with bike data
✓ Loaded karlsruhe: 26,304 hours, 26,031 with bike data
✓ Loaded ludwigsburg: 26,304 hours, 26,228 with bike data
✓ Loaded freiburg: 26,304 hours, 26,230 with bike data
✓ Loaded ulm: 26,304 hours, 25,490 with bike data
✓ Loaded ravensburg: 26,304 hours, 26,266 with bike data
✓ Loaded loerrach: 26,304 hours, 26,266 with bike data
✓ Loaded konstanz: 26,304 hours, 26,228 with bike data
✓ Loaded heilbronn: 26,304 hours, 26,266 with bike data
✓ Loaded kirchheim: 26,304 hours, 26,228 with bike data

Total: 13 cities loaded


## 2. Single City Analysis: Weather Index Visualization

Let's examine how bike traffic relates to weather conditions at a single station.

In [4]:
if USE_CITIES:
    # Pick one city for detailed analysis
    city = "tuebingen"
    station_alias = city
    df = data_dict[city]
    display_name = f"City {city.capitalize()}"

    # Calculate weather index
    df_with_index = add_weather_index(df)
else:
    # Pick one station for detailed analysis
    station_alias = "tuebingen_tunnel"
    df = data_dict[station_alias]
    station = get_station(station_alias)
    display_name = station.display_name

    # Calculate weather index
    df_with_index = add_weather_index(df)

print(f"Weather Index Statistics for {station_alias}:")
print(f"  Mean: {df_with_index['weather_index'].mean():.3f}")
print(f"  Std:  {df_with_index['weather_index'].std():.3f}")
print(f"  Q1:   {df_with_index['weather_index'].quantile(0.25):.3f}")
print(f"  Q3:   {df_with_index['weather_index'].quantile(0.75):.3f}")

Weather Index Statistics for tuebingen:
  Mean: 0.383
  Std:  0.148
  Q1:   0.277
  Q3:   0.465


In [5]:
# Scatter plot: Bike counts vs Weather Index
fig = plot_weather_index_scatter(
    df, 
    title=display_name,
    hour_range=(6, 23),
    weekdays_only=True
)
fig.show()

In [6]:
# Box plot: Bike counts by weather quartile
fig = plot_weather_quartile_boxplot(
    df,
    title=display_name,
    hour_range=(6, 9),
    weekdays_only=True
)
fig.show()

In [7]:
# Hourly pattern by weather quality
fig = plot_weather_index_hourly(
    df,
    title=display_name,
    weekdays_only=True,
)
fig.show()

## 3. Multi-City Comparison: Weather Elasticity

Now let's compare how different cities respond to weather changes. Cities with higher resilience (elasticity closer to 0) are considered to have a stronger "bicycle culture."

In [8]:
# Calculate elasticity for all cities
elasticity_results = []

for alias, df in data_dict.items():
    try:
        df_idx = add_weather_index(df)
        result = estimate_city_elasticity(df_idx)
        result['station'] = alias
        elasticity_results.append(result)
    except Exception as e:
        print(f"Error for {alias}: {e}")

# Create summary table
summary_df = pd.DataFrame(elasticity_results)
summary_df = summary_df.sort_values('elasticity', ascending=False)

print("Weather Elasticity Summary (sorted by resilience):")
print("="*60)
for _, row in summary_df.iterrows():
    sig = "***" if row['pvalue'] < 0.001 else "**" if row['pvalue'] < 0.01 else "*" if row['pvalue'] < 0.05 else "ns"
    print(f"{row['station']:30} {row['elasticity']:7.2f}% {sig:4} (R²={row['r_squared']:.4f}, n={row['n_obs']:,})")

Weather Elasticity Summary (sorted by resilience):
freiburg                         -0.37% ***  (R²=0.0177, n=11,958)
ludwigsburg                      -0.50% ***  (R²=0.0474, n=10,704)
mannheim                         -0.51% ***  (R²=0.0362, n=11,721)
loerrach                         -0.53% ***  (R²=0.0390, n=11,322)
heidelberg                       -0.54% ***  (R²=0.0297, n=11,841)
stuttgart                        -0.65% ***  (R²=0.0931, n=10,826)
konstanz                         -0.65% ***  (R²=0.0535, n=11,694)
ulm                              -0.65% ***  (R²=0.1079, n=9,843)
karlsruhe                        -0.67% ***  (R²=0.0456, n=11,557)
tuebingen                        -0.68% ***  (R²=0.0496, n=11,872)
kirchheim                        -0.72% ***  (R²=0.0646, n=10,416)
ravensburg                       -0.73% ***  (R²=0.0892, n=10,696)
heilbronn                        -0.89% ***  (R²=0.1416, n=11,082)


In [9]:
# Bar chart comparing weather elasticity across cities
fig = plot_city_elasticity_comparison(
    data_dict,
    title="Weather Elasticity Comparison Across Cities",
    hour_range=(6, 9),
    weekdays_only=True
)
fig.show()

In [10]:
# Bar chart comparing weather elasticity across cities
fig = plot_city_elasticity_comparison(
    data_dict,
    title="Weather Elasticity Comparison Across Cities",
    hour_range=(6, 23),
    weekend_only=True,
    quantile_a=0.25,
    quantile_b=0.75,
)
fig.show()

In [11]:
# Heatmap showing hourly weather sensitivity by city
fig = plot_city_weather_sensitivity_heatmap(
    data_dict,
    title="Hourly Weather Sensitivity by City",
    hour_range=(6, 9),
    weekdays_only=True
)
fig.show()

In [12]:
# Comprehensive city resilience ranking
fig = plot_city_resilience_ranking(
    data_dict,
    title="City Weather Resilience Ranking",
    hour_range=(6, 9),
    weekdays_only=True
)
fig.show()

## 4. Interpretation

### Key Findings

- **Weather Elasticity** measures how much cycling drops when weather worsens (Q1→Q3 shift)
- Cities with elasticity **closer to 0** have more resilient cyclists
- Cities with **more negative elasticity** have "fair-weather cyclists" who avoid bad weather

### Methodology Notes

Based on Goldmann & Wessel (2021):
- The composite weather index uses geometric aggregation (not linear averaging)
- Temperature is inverted so that colder = worse weather
- Extreme values are winsorized at the 99.8th percentile
- The final index is normalized to [0, 1]

### Potential Policy Implications

1. **Infrastructure**: Cities with poor weather resilience may benefit from better cycling infrastructure (sheltered routes, better drainage)
2. **Public Transport**: Fair-weather cyclists are likely to switch modes - transit systems could offer "bad weather tickets"
3. **Flexible Lane Usage**: In sensitive cities, bike lanes could potentially be shared during severe weather

## 5. Data Quality and Methodology Notes

### Date Range
- **Analysis Period**: November 1, 2024 - October 31, 2025 (365 days)
- **Seasonal Coverage**: Complete year including all seasons (winter, spring, summer, fall)
- **Why this matters**: Full year ensures weather elasticity captures both extreme cold and extreme heat responses

### Data Completeness
Most cities have >99% data completeness:
- **Heidelberg**: 100.0% complete (8,760/8,760 hours)
- **Mannheim**: 99.9% complete (8,747/8,760 hours)  
- **Stuttgart**: 99.4% complete (8,711/8,760 hours)

### How Missing Data is Handled

**1. Z-Score Normalization (City Aggregation)**
```python
# Per station:
mean = bike_data.mean()  # Calculated only on non-NaN values
std = bike_data.std()     # Calculated only on non-NaN values
z_score = (bike_data - mean) / std

# Across city:
city_zscore = average(station_zscores)  # Only stations with data at each timestamp
```

**Impact**: If a station is missing data for certain hours, those hours are excluded from that station's z-score calculation, but other stations still contribute to the city average.

**2. Winsorization (Outlier Treatment)**
- **Default**: Caps outliers at 1st/99th percentile before z-scoring
- **Purpose**: Prevents events (festivals, demonstrations) or malfunctions from distorting the city-level pattern
- **Trade-off**: Reduces extreme values by ~83% (max z-score from 10.5 to 1.8)

**3. Weather Elasticity Calculation**
```python
df_clean = df.dropna(subset=['bike', 'weather_index'])  # Removes NaN rows
```

**Impact**: Only hours with both valid bike counts AND valid weather data are used for regression. With >99% completeness, this has minimal impact (~50 hours out of 8,760).

### Potential Biases

1. **Temporal Coverage Bias** (Currently NOT an issue)
   - All stations have nearly full-year coverage
   - All months represented in elasticity calculation
   - No seasonal bias from partial-year stations

2. **Zero Values**
   - Heidelberg has 51 hours (0.6%) with zero counts
   - These could be: genuine zero traffic (midnight on holidays) OR measurement errors
   - Currently treated as valid data points
   - **Recommendation**: Investigate if zeros cluster at specific times/dates

3. **Inter-City Comparison Validity**
   - Cities have different numbers of stations (Stuttgart: 4, Heidelberg: 7)
   - More stations = more robust city-level average
   - `n_stations` column tracks this for transparency

4. **Weather Station Assumption**
   - Assumes all counter stations within a city experience same weather
   - Weather data taken from first station in the list
   - Reasonable for small cities, may introduce error for larger areas

In [13]:
# Data Quality Visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Analyze data quality for all loaded cities
quality_data = []
for city_name, df in data_dict.items():
    total = len(df)
    valid = df['bike'].notna().sum()
    zeros = (df['bike'] == 0).sum()
    n_stations_avg = df['n_stations'].mean()
    n_stations_min = df['n_stations'].min()
    
    quality_data.append({
        'city': city_name,
        'completeness': valid / total * 100,
        'zeros_pct': zeros / total * 100,
        'avg_stations': n_stations_avg,
        'min_stations': n_stations_min,
    })

quality_df = pd.DataFrame(quality_data).sort_values('completeness', ascending=False)

# Create visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Data Completeness', 'Station Coverage'),
    specs=[[{"type": "bar"}, {"type": "bar"}]]
)

# Completeness chart
fig.add_trace(
    go.Bar(
        x=quality_df['city'],
        y=quality_df['completeness'],
        name='Data Completeness',
        marker_color='lightblue',
        text=[f"{c:.1f}%" for c in quality_df['completeness']],
        textposition='outside',
    ),
    row=1, col=1
)

fig.add_hline(y=99.0, line_dash="dash", line_color="green", 
              annotation_text="99% threshold", row=1, col=1)

# Station coverage chart
fig.add_trace(
    go.Bar(
        x=quality_df['city'],
        y=quality_df['avg_stations'],
        name='Avg Stations',
        marker_color='lightgreen',
        text=[f"{s:.1f}" for s in quality_df['avg_stations']],
        textposition='outside',
    ),
    row=1, col=2
)

fig.update_xaxes(title_text="City", tickangle=-45, row=1, col=1)
fig.update_xaxes(title_text="City", tickangle=-45, row=1, col=2)
fig.update_yaxes(title_text="Completeness [%]", range=[98, 101], row=1, col=1)
fig.update_yaxes(title_text="Number of Stations", row=1, col=2)

fig.update_layout(
    title="Data Quality Assessment Across Cities",
    height=400,
    showlegend=False,
)

fig.show()

# Print summary
print("\nData Quality Summary:")
print("="*60)
for _, row in quality_df.iterrows():
    print(f"{row['city']:15} {row['completeness']:5.1f}% complete, "
          f"{row['avg_stations']:4.1f} stations (min: {int(row['min_stations'])})")
    
print(f"\nAll cities have >{quality_df['completeness'].min():.1f}% completeness ✓")


Data Quality Summary:
stuttgart        99.9% complete,  9.0 stations (min: 0)
mannheim         99.9% complete,  7.8 stations (min: 0)
ravensburg       99.9% complete,  3.9 stations (min: 0)
loerrach         99.9% complete,  2.0 stations (min: 0)
heilbronn        99.9% complete,  2.8 stations (min: 0)
heidelberg       99.8% complete,  6.5 stations (min: 0)
freiburg         99.7% complete,  2.9 stations (min: 0)
tuebingen        99.7% complete,  2.0 stations (min: 0)
ludwigsburg      99.7% complete,  2.0 stations (min: 0)
konstanz         99.7% complete,  1.0 stations (min: 0)
kirchheim        99.7% complete,  1.0 stations (min: 0)
karlsruhe        99.0% complete,  1.0 stations (min: 0)
ulm              96.9% complete,  1.0 stations (min: 0)

All cities have >96.9% completeness ✓
